# ScamLens Phase 3.6 - train on Colab / Kaggle

**Never commit model weights or checkpoints.** `*.pt`, `*.safetensors`, `checkpoints/` and `outputs/` are gitignored. The last cell zips only logs, configs, predictions and results.

Runtime -> Change runtime type -> **GPU (T4 is enough)**. Estimated GPU time on a free T4: **E1 ~50 min**, **E1s ~35 min**, optional **E2 ~40 min**, optional **E3 ~55 min**. A full data rebuild downloads public data and takes ~10-20 min; the Kaggle bundle skips it.

**Two ways in:** on Kaggle, attach `scamlens_bundle.zip` as a dataset (Add Data -> Your datasets) and this notebook copies it into the working directory. Everywhere else it git-clones the repo and rebuilds the data.

**Checkpoint selection (Phase 3.6):** the best epoch is the *mean of `val_synthetic` macro-F1 and `val_real` balanced accuracy*. Real val is 98.3% scam, so combined-val macro-F1 is logged for observation only - it used to reward over-flagging. No test split ever selects an epoch.


In [ ]:
# 1. Get the project.
#    Kaggle: attach the bundle as a dataset (Add Data -> scamlens-bundle).
#    Otherwise: git clone (edit REPO_URL) and rebuild the data in step 4.
import os, shutil
from pathlib import Path

BUNDLE_ROOT = Path("/kaggle/input/scamlens-bundle")
REPO_URL = "https://github.com/YOUR_USERNAME/scamlens.git"
WORK = Path.cwd() / "scamlens"

if BUNDLE_ROOT.is_dir():
    # The bundle already contains src/, configs/, tests/, notebooks/, README.md
    # and data/processed/messages.parquet, so nothing needs downloading.
    if not WORK.exists():
        shutil.copytree(BUNDLE_ROOT, WORK)
    os.chdir(WORK)
    print("Using the Kaggle bundle at", WORK)
else:
    if not WORK.exists():
        !git clone $REPO_URL scamlens
    os.chdir(WORK)
    print("Cloned from git at", WORK)

print("messages.parquet present:", Path("data/processed/messages.parquet").exists())


In [ ]:
# 2. Install current dependencies (Phase 3 packages are deliberately UNPINNED).
!python -m pip install -q -r requirements.txt

In [ ]:
# 3. Check the GPU. If CUDA is False, switch the runtime type and rerun.
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
    print("memory GB:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
    !nvidia-smi
else:
    print("WARNING: training on CPU is 8-20x slower. Do not start E1 here.")

## Data + baselines (CPU, one-time)

The bundle already ships `data/processed/messages.parquet` with the cluster-safe splits, so the expensive rebuild is skipped. On a bare git clone the notebook runs the whole honest pipeline: synthetic generator -> real public rows split by near-duplicate cluster -> dedup + tag-safe splits -> checks -> nearest-neighbour leakage audit -> baselines -> evaluation.

The build fails loudly if any tactic tag has fewer than 100 train positives or no val/test_b positive. `test_a` is 10% of the TRAIN clusters (both classes); `test_b` is unseen families; `test_real` is never blended.


In [ ]:
# 4. Data + baselines (CPU, one-time).
from pathlib import Path

if Path("data/processed/messages.parquet").exists():
    print("messages.parquet already present (bundle) - skipping the rebuild")
else:
    !python -m src.synth.generate
    !python -m src.external_data
    !python -m src.build_dataset
    !python -m src.data_checks

!python -m src.nn_sim        # max similarity to any TRAIN row -> results/nn_sim.parquet
!python -m src.baselines     # TF-IDF baselines (models/ is excluded from the bundle)
!python -m src.audit         # shortcut audit, appended to RESULTS.md
!python -m src.evaluate      # writes results/RESULTS.md + metrics.json


In [ ]:
# 5. Smoke test: 16 examples, 30 steps. MUST print PASS before real training.
#    The encoder uses lr_encoder=3e-5; the new heads use lr_heads=1e-3 (configs/train.yaml).
!python -m src.overfit_smoke_test --steps 30


## Verify the model ids and the real-data gate before spending GPU time

`--dry_run` downloads each model config (proof the id exists), checks the E4 gate and prints every command plus its estimated T4 time. It never guesses an id.


In [ ]:
# 6. Dry run.
!python -m src.run_experiments --dry_run

## E1 -> E1s (E2/E3 optional, E4 only if the gate passes)

Two mixes are trained by default: **E1 = MuRIL + tags on synth+real** (the main model) and **E1s = MuRIL + tags on synth-only** (shows the value of real data). Both use the same validation-only selection score.

Checkpoint selection = **mean(`val_synthetic` macro-F1, `val_real` balanced accuracy)**. The log prints both numbers, the combined-val macro-F1 (observation only) and every test split. `test_a`, `test_b` and each `test_real` stratum never choose an epoch. E2 (verdict-only) and E3 (XLM-R) are OFF by default; add `--include_optional`.


In [ ]:
# 7. Run the experiments (E1 + E1s). If the runtime disconnects, rerun with --skip_existing.
!python -m src.run_experiments --skip_existing

# Optional ablations (verdict-only MuRIL, XLM-R):
# !python -m src.run_experiments --skip_existing --include_optional


In [ ]:
# 8. Refresh the comparison tables, tokenizer report and CPU benchmark.
!python -m src.evaluate
!python -m src.tokenizer_inspect
!python -m src.benchmark


In [ ]:
# 9. Zip ONLY what a reviewer needs: predictions, logs, configs and results.
#    Weights (*.pt / *.bin / *.safetensors), tokenizer dumps and __pycache__
#    are excluded on purpose - they are large and reproducible.
import zipfile
from pathlib import Path

OUT = Path("scamlens_outputs.zip")
SKIP_SUFFIX = {".pt", ".pth", ".bin", ".safetensors", ".ckpt", ".h5"}

def keep(path: Path) -> bool:
    if path.suffix in SKIP_SUFFIX:
        return False
    if "tokenizer" in path.parts or "__pycache__" in path.parts:
        return False
    return True

with zipfile.ZipFile(OUT, "w", zipfile.ZIP_DEFLATED) as zf:
    for folder in ("outputs", "results", "configs"):
        if not Path(folder).exists():
            continue
        for path in Path(folder).rglob("*"):
            if path.is_file() and keep(path):
                # outputs/<run>/preds_*.parquet, training_log.jsonl,
                # config_used.yaml, best_epoch.json, run_meta.json land here.
                zf.write(path, path.as_posix())

print("zip MB:", round(OUT.stat().st_size / 1e6, 1))
try:
    from google.colab import files
    files.download(str(OUT))                     # Colab
except ImportError:
    print("Kaggle: download", OUT.name, "from the Output panel")   # Kaggle


## Reminder

- Do **not** commit `outputs/`, `*.pt`, `*.safetensors` or `checkpoints/`. They are gitignored for a reason.
- The zip contains `preds_*.parquet` so Phase 4 can do calibration and error analysis without the weights.
- To retrain from scratch, delete `outputs/<run_name>/` and rerun that experiment.